# Exercise 5.1 — Passing values to the shader with uniforms

A **uniform** is a value that is the same for every vertex of one draw call. We set it from JavaScript with
`gl.uniform1f` (one float) or `gl.uniform3f` (three floats), and the shader reads it.

Here the vertex shader receives:
* `zcoord` (float) — becomes the z of every vertex,
* `ycoord` (float) — added to the y of every vertex,
* `color` (vec3) — the colour of all vertices (instead of the colour buffer).

In `render()` we draw the **same triangle twice**:
1. `zcoord = 0`, `ycoord = 0`, colour red → draw
2. `zcoord = sin(time/1000)`, `ycoord = 0.2`, colour blue → draw again

Because the blue `z` goes from −1 to +1, the blue triangle is sometimes in front of the red one and sometimes
behind it (in NDC a **smaller z is closer** to us).

**Depth test.** Press **D** (after clicking the canvas) to switch `gl.enable(gl.DEPTH_TEST)` / `gl.disable(...)`:
* enabled: the GPU keeps, for each pixel, the closest z, so the triangles swap front/back correctly;
* disabled: the last triangle drawn (blue) is always on top, whatever its z.

*Footnote question of the sheet — "does this go against the definition of projection?"* A projection should
reduce dimension (3D → 2D). OpenGL keeps a z value after the projection, but only to decide visibility; the 2D
image is obtained by dropping z after dividing by w. So the 2D projection is still there; the extra z is just
kept for the depth test.

## Library
Only **glm-js** is needed (for matrices). The WebGL code is written directly, so this notebook still works
even when the course JS server is down.

In [ ]:
%%html
<script src='https://git.io/glm-js.min.js'></script>

In [ ]:
%%html
<script id="ex51-vs" type="x-shader/x-vertex">
precision mediump float;
attribute vec3 in_Position;
uniform float zcoord;     // replaces the z of the vertices
uniform float ycoord;     // added to the y of the vertices
void main(void) {
    gl_Position = vec4(in_Position.x, in_Position.y + ycoord, zcoord, 1.0);
}
</script>

In [ ]:
%%html
<script id="ex51-fs" type="x-shader/x-fragment">
precision mediump float;
uniform vec3 color;       // one colour for the whole triangle
void main(void) {
    gl_FragColor = vec4(color, 1.0);
}
</script>

In [ ]:
%%html
<script>
// ---------- small WebGL helpers ----------
// compile the two shaders (read from the <script> tags) and link them into a program
window.makeProgram = function (gl, vsId, fsId) {
  const compile = (type, id) => {
    const s = gl.createShader(type);
    gl.shaderSource(s, document.getElementById(id).text);
    gl.compileShader(s);
    if (!gl.getShaderParameter(s, gl.COMPILE_STATUS)) console.error(id, gl.getShaderInfoLog(s));
    return s;
  };
  const p = gl.createProgram();
  gl.attachShader(p, compile(gl.VERTEX_SHADER, vsId));
  gl.attachShader(p, compile(gl.FRAGMENT_SHADER, fsId));
  gl.linkProgram(p);
  return p;
};
// copy a JS array of numbers into a GPU buffer (GL wants 32-bit floats)
window.makeBuffer = function (gl, data) {
  const b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
};
// connect a buffer (3 floats per vertex) to an attribute of the shader
window.useAttrib = function (gl, prog, name, buffer) {
  const loc = gl.getAttribLocation(prog, name);
  if (loc < 0) return;
  gl.bindBuffer(gl.ARRAY_BUFFER, buffer);
  gl.vertexAttribPointer(loc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(loc);
};
// call render(time) on every animation frame
window.animate = function (render) {
  const step = (time) => { render(time); requestAnimationFrame(step); };
  requestAnimationFrame(step);
};
</script>

In [ ]:
%%html
<canvas id="ex51Canvas" width="500" height="500" style="border:2px solid #000000;"></canvas>
<div id="ex51CanvasInfo" style="font-family:monospace"></div>
<script>
(function () {
  const cv = document.getElementById("ex51Canvas");
  const gl = cv.getContext("webgl");
  const prog = makeProgram(gl, "ex51-vs", "ex51-fs");
  const tri = makeBuffer(gl, [-0.6, -0.6, 0,   0.6, -0.6, 0,   -0.6, 0.6, 0]);

  // where the uniforms live inside the program
  const uZ = gl.getUniformLocation(prog, "zcoord");
  const uY = gl.getUniformLocation(prog, "ycoord");
  const uC = gl.getUniformLocation(prog, "color");

  let depth = true;
  gl.enable(gl.DEPTH_TEST);
  gl.clearColor(1, 1, 1, 1);

  cv.setAttribute("tabindex", "0");
  cv.addEventListener("click", () => cv.focus());
  cv.addEventListener("keydown", (e) => {
    e.stopPropagation();
    if (e.key === "d" || e.key === "D") {
      depth = !depth;
      if (depth) gl.enable(gl.DEPTH_TEST); else gl.disable(gl.DEPTH_TEST);
    }
  });

  animate((time) => {
    gl.clear(gl.COLOR_BUFFER_BIT | gl.DEPTH_BUFFER_BIT);
    gl.useProgram(prog);
    useAttrib(gl, prog, "in_Position", tri);

    // 1. red triangle, z = 0
    gl.uniform1f(uZ, 0.0);
    gl.uniform1f(uY, 0.0);
    gl.uniform3f(uC, 1.0, 0.0, 0.0);
    gl.drawArrays(gl.TRIANGLES, 0, 3);

    // 2. blue triangle, moved up 0.2, z goes from -1 to 1
    const z = Math.sin(time / 1000);
    gl.uniform1f(uZ, z);
    gl.uniform1f(uY, 0.2);
    gl.uniform3f(uC, 0.0, 0.0, 1.0);
    gl.drawArrays(gl.TRIANGLES, 0, 3);

    document.getElementById("ex51CanvasInfo").textContent =
      "depth test: " + (depth ? "ON" : "OFF") + " | blue z = " + z.toFixed(2) +
      (z < 0 ? " (blue in front)" : " (blue behind)") + " | click canvas, press D";
  });
})();
</script>